In [0]:
%pip install torch>=2.0.0 torchvision>=0.15.0 xgboost fastf1>=3.6.0 scikit-learn pandas numpy

In [0]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import pandas as pd
import os
import numpy as np

In [0]:
class BasePaceModel:
    def __init__(self, df):
        self.df = df
        self.features = []
        self.target = 'LapTime_sec'
        self.X = None
        self.y = None
        self.model = None 
        self.mae = None
        
        self._define_features()
        self._clean_data()
    def _define_features(self):
        base_features = [
            'LapNumber', 'Compound', 'TyreLife', 'AirTemp', 'Humidity', 
            'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed',
             'Circuit_CircuitLength', 'Circuit_Number_of_Laps',
             'Circuit_NumberOfTurns', 'Circuit_AverageAngleAbs',
            'Circuit_AverageAngle',
            'GapToLeader', 'GapToAhead', 'GapToBehind',
            'status_1','status_12','status_124','status_21','status_24','status_4','status_41',
            'TimeSinceLastWeatherMeasurement',
        ]

        # Start with base features
        self.X = self.df[base_features].copy()
        
        # One-hot encode Driver_idx
        if 'Driver_idx' in self.df.columns:
            driver_dummies = pd.get_dummies(self.df['Driver_idx'], prefix='Driver', dtype=int)
            self.X = pd.concat([self.X, driver_dummies], axis=1)
            print(f"Added {len(driver_dummies.columns)} one-hot encoded driver features")
        
        # One-hot encode Team_idx
        if 'Team_idx' in self.df.columns:
            team_dummies = pd.get_dummies(self.df['Team_idx'], prefix='Team', dtype=int)
            self.X = pd.concat([self.X, team_dummies], axis=1)
            print(f"Added {len(team_dummies.columns)} one-hot encoded team features")
        
        # Update feature list with all column names
        self.features = self.X.columns.tolist()
        self.y = self.df[self.target].copy()
        
        print(f"Defined {len(self.features)} total features")

    def _clean_data(self):
        print(f"\nCleaning data...")
        print(f"Initial shape: X={self.X.shape}, y={self.y.shape}")
        

        mask = ~self.y.isna()
        rows_before = len(self.y)
        self.X = self.X[mask]
        self.y = self.y[mask]
        self.df = self.df[mask]
        rows_dropped = rows_before - len(self.y)
        print(f"Dropped {rows_dropped} rows with NaN target ({rows_dropped/rows_before*100:.1f}%)")
        
        nan_counts = self.X.isna().sum()
        if nan_counts.sum() > 0:
            print(f"\nNaN values in features before filling:")
            print(nan_counts[nan_counts > 0])
        other_cols = [col for col in self.X.columns]
        
        
        if other_cols and self.X[other_cols].isna().sum().sum() > 0:
            self.X[other_cols] = self.X[other_cols].fillna(self.X[other_cols].median())
            print(f"Filled {len(other_cols)} non-embedding features with median")
        
        print(f"\nData cleaning complete.")
        print(f"Final shape: X={self.X.shape}, y={self.y.shape}")
        print(f"Remaining NaN - X: {self.X.isna().sum().sum()}, y: {self.y.isna().sum()}")

    def prepare_dataframe_for_prediction(self, df):
        """
        Prepare any dataframe for prediction using the SAME features as training.
        Handles one-hot encoding consistently for drivers/teams.
        """
        base_features = [
            'LapNumber', 'Compound', 'TyreLife', 'AirTemp', 'Humidity', 
            'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed',
             'Circuit_CircuitLength', 'Circuit_Number_of_Laps',
            'Circuit_NumberOfTurns', 'Circuit_AverageAngleAbs',
             'Circuit_AverageAngle',
            'GapToLeader', 'GapToAhead', 'GapToBehind',
            'status_1','status_12','status_124','status_21','status_24','status_4','status_41',
            'TimeSinceLastWeatherMeasurement',
        ]
        
        # Start with base features
        X_new = df[base_features].copy()
        
        # One-hot encode Driver_idx
        if 'Driver_idx' in df.columns:
            driver_dummies = pd.get_dummies(df['Driver_idx'], prefix='Driver', dtype=int)
            X_new = pd.concat([X_new, driver_dummies], axis=1)
        
        # One-hot encode Team_idx
        if 'Team_idx' in df.columns:
            team_dummies = pd.get_dummies(df['Team_idx'], prefix='Team', dtype=int)
            X_new = pd.concat([X_new, team_dummies], axis=1)
        
        # Align columns with training features
        # Add missing columns (e.g., drivers in training but not in new data)
        for col in self.features:
            if col not in X_new.columns:
                X_new[col] = 0
        
        # Keep only training features (drop extra columns from new data)
        X_new = X_new[self.features]
        
        # Fill NaN values
        if X_new.isna().sum().sum() > 0:
            X_new = X_new.fillna(X_new.median())
        
        print(f"Prepared {len(df)} rows for prediction with {X_new.shape[1]} features")
        
        return df, X_new

    def train(self, test_size=0.3, random_state=42):
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=test_size, random_state=random_state
        )
        
        self.model = xgb.XGBRegressor(
    n_estimators=100,        # Reduced from 300 - fewer trees
    learning_rate=0.03,      # Reduced from 0.05 - slower learning
    max_depth=2,             # Reduced from 3 - shallower trees
    min_child_weight=10,     # Increased from 5 - more conservative splits
    subsample=0.6,           # Reduced from 0.7 - more aggressive row sampling
    colsample_bytree=0.6,    # Reduced from 0.7 - more aggressive feature sampling
    reg_alpha=2.0,           # Increased from 0.5 - stronger L1 regularization
    reg_lambda=3.0,          # Increased from 1.0 - stronger L2 regularization
    random_state=42,
    tree_method='hist',
    early_stopping_rounds=20  # Reduced from 50 - stop earlier
)
        print("Training model...")
        self.model.fit(
            self.X_train, self.y_train,
            eval_set=[(self.X_test, self.y_test)],
            verbose=50
        )
        
    def test(self):
        test_predictions = self.model.predict(self.X_test)
        self.mae = mean_absolute_error(self.y_test, test_predictions)
        print(f"\nModel training complete. Validation MAE: {self.mae:.3f} seconds")
        
        feature_importance = pd.DataFrame({
            'feature': self.features,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print("\nTop 10 Most Important Features:")
        print(feature_importance.head(10))
        

        
    def save(self, model_path='models/base_pace_model.txt'):
        if self.model is None:
            raise ValueError("Model has not been trained yet. Call .train() before saving.")
        os.makedirs(os.path.dirname(model_path), exist_ok=True)
        self.model.save_model(model_path)
        print(f"Model saved successfully to {model_path}")

    def load(self, model_path='models/base_pace_model.txt'):
        self.model = xgb.XGBRegressor()
        self.model.load_model(model_path)
        print(f"Model loaded successfully from {model_path}")

In [0]:
def label_modes_with_confidence(df, model, X, confidence_threshold=1.5, mae=0.484, target='LapTime_sec'):
    if model is None:
        raise ValueError("Model not found. Train a new model with .train() or load one with .load().")
    
    predicted_pace = model.predict(X)
    delta = df[target].values - predicted_pace
    
    threshold = confidence_threshold * mae
    conditions = [
        delta < -threshold,  # Push
        delta > threshold    # Conserve
    ]
    choices = [2, 0]
    
    labeled_df = df.copy()
    labeled_df['mode'] = np.select(conditions, choices, default=1)
    labeled_df['predicted_base_pace'] = predicted_pace
    labeled_df['delta'] = delta
    
    print("\n=== Labeling Statistics ===")
    print(f"Total laps: {len(labeled_df)}")
    print(f"Mode distribution:")
    mode_counts = labeled_df['mode'].value_counts().sort_index()
    mode_names = {0: 'conserve', 1: 'base', 2: 'push'}
    for mode_num, count in mode_counts.items():
        print(f"  {mode_names[mode_num]}: {count}")
    
    return labeled_df

In [0]:
if __name__ == "__main__":
    from pyspark.sql import SparkSession
    
    # Configuration: Set to True to retrain model even if it exists
    FORCE_RETRAIN = True
    MODEL_PATH = '../models/base_pace_model.txt'
    
    # Initialize Spark session
    spark = SparkSession.builder.getOrCreate()
    
    # Read validation data from Unity Catalog
    print("Reading raw_validating data from Unity Catalog...")
    df1 = spark.table("workspace.f1_racing_laptime_pred.raw_validating").toPandas()
    
    # Load training data (already normalized)
    print("Reading training data (normalized)...")
    df_train = spark.table("workspace.f1_racing_laptime_pred.raw_training").toPandas()
    
    # Load the scalers to calculate normalized threshold
    import joblib
    import numpy as np
    scalers = joblib.load('/Workspace/Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/feature engineering/training_temp_scalers.joblib')
    
    # Calculate normalized threshold for 2.0 seconds
    # Gaps were transformed with log1p + StandardScaler:
    # 1. Apply log1p: log(1 + 2.0)
    # 2. Apply StandardScaler: (value - mean) / std
    gap_cols = ['GapToAhead', 'GapToBehind']
    skewed_scalers = scalers.get('skewed', {})
    
    threshold_original = 2.0  # seconds
    normalized_thresholds = {}
    
    for col in gap_cols:
        if col in skewed_scalers:
            # Step 1: Apply log1p transform
            log_threshold = np.log1p(threshold_original)
            # Step 2: Apply StandardScaler
            scaler = skewed_scalers[col]
            normalized_threshold = (log_threshold - scaler.mean_[0]) / scaler.scale_[0]
            normalized_thresholds[col] = normalized_threshold
            print(f"  {col}: 2.0s → normalized threshold = {normalized_threshold:.4f}")
    
    # Filter using normalized thresholds (data is already normalized)
    print(f"Filtering for base pace laps (normalized gaps >= {threshold_original}s)...")
    df_base_pace = df_train[
        (df_train['GapToAhead'] >= normalized_thresholds['GapToAhead']) & 
        (df_train['GapToBehind'] >= normalized_thresholds['GapToBehind']) & 
        (df_train['dnf'] == 0) & 
        (df_train['status_1'] == 1)
    ].copy()
    
    print(f"Base pace laps: {len(df_base_pace)}")
    
    # Check if model exists and decide whether to train or load
    model_exists = os.path.exists(MODEL_PATH)
    
    if model_exists and not FORCE_RETRAIN:
        print(f"\n{'='*60}")
        print(f"Existing model found at {MODEL_PATH}")
        print(f"Loading existing model instead of retraining...")
        print(f"(Set FORCE_RETRAIN=True to retrain)")
        print(f"{'='*60}\n")
        
        # Create a BasePaceModel instance with base pace data (needed for embeddings)
        obj = BasePaceModel(df_base_pace)
        # Load the pre-trained model
        obj.load(MODEL_PATH)
    else:
        if FORCE_RETRAIN:
            print(f"\n{'='*60}")
            print(f"FORCE_RETRAIN=True: Training new model...")
            print(f"{'='*60}\n")
        else:
            print(f"\n{'='*60}")
            print(f"No existing model found. Training new model...")
            print(f"{'='*60}\n")
        
        # Train model on base pace laps
        obj = BasePaceModel(df_base_pace)
        obj.train()
        obj.test()
        obj.save(MODEL_PATH)
    
    # === VALIDATION DATA EVALUATION ===
    print("\n" + "="*60)
    print("VALIDATION DATA EVALUATION")
    print("="*60)
    
    # Filter validation data for base pace laps using the same normalized thresholds
    df_val_base_pace = df1[
        (df1['GapToAhead'] >= normalized_thresholds['GapToAhead']) & 
        (df1['GapToBehind'] >= normalized_thresholds['GapToBehind']) & 
        (df1['dnf'] == 0) & 
        (df1['status_1'] == 1)
    ].copy()
    
    print(f"\nValidation base pace laps: {len(df_val_base_pace)} / {len(df1)} ({len(df_val_base_pace)/len(df1)*100:.1f}%)")
    
    if len(df_val_base_pace) > 0:
        obj_val = BasePaceModel(df_val_base_pace)
        
        # Get the lap time scaler for inverse transformation
        lap_scaler = scalers['skewed']['LapTime_sec']
        
        # === TRAINING TEST SET DENORMALIZATION ===
        # Make predictions on TEST SPLIT ONLY (to match obj.mae calculation)
        test_predictions_norm = obj.model.predict(obj.X_test)
        
        # Denormalize test predictions: inverse StandardScaler, then inverse log1p
        test_predictions_log = test_predictions_norm * lap_scaler.scale_[0] + lap_scaler.mean_[0]
        test_predictions_original = np.expm1(test_predictions_log)
        
        # Denormalize test actuals
        test_actuals_log = obj.y_test.values * lap_scaler.scale_[0] + lap_scaler.mean_[0]
        test_actuals_original = np.expm1(test_actuals_log)
        
        # Calculate MAE in original space (on test split)
        test_mae_original = mean_absolute_error(test_actuals_original, test_predictions_original)
        
        # === VALIDATION SET DENORMALIZATION ===
        # Make predictions on validation data in normalized space
        val_predictions_norm = obj.model.predict(obj_val.X)
        
        # Denormalize validation predictions
        val_predictions_log = val_predictions_norm * lap_scaler.scale_[0] + lap_scaler.mean_[0]
        val_predictions_original = np.expm1(val_predictions_log)
        
        # Denormalize validation actuals
        val_actuals_log = obj_val.y.values * lap_scaler.scale_[0] + lap_scaler.mean_[0]
        val_actuals_original = np.expm1(val_actuals_log)
        
        # Calculate MAE in original space
        val_mae_original = mean_absolute_error(val_actuals_original, val_predictions_original)
        
        # Also calculate normalized MAE for comparison
        val_mae_norm = mean_absolute_error(obj_val.y, val_predictions_norm)
        
        # Get original lap time statistics for context
        lap_mean_original = scalers['lap_mean']
        lap_std_original = scalers['lap_std']
        
        print(f"\n{'='*60}")
        print("MODEL PERFORMANCE COMPARISON")
        print("="*60)
        print(f"\nNormalized MAE (standardized scale):")
        print(f"  Training (test split): {obj.mae:.3f}")
        print(f"  Validation:            {val_mae_norm:.3f}")
        print(f"  Gap:                   {abs(val_mae_norm - obj.mae):.3f}")
        print(f"\nDenormalized MAE (actual seconds):")
        print(f"  Training (test split): {test_mae_original:.3f} seconds ({test_mae_original/lap_mean_original*100:.2f}% error)")
        print(f"  Validation:            {val_mae_original:.3f} seconds ({val_mae_original/lap_mean_original*100:.2f}% error)")
        print(f"  Gap:                   {abs(val_mae_original - test_mae_original):.3f} seconds")
        print(f"\nOriginal lap time context: mean={lap_mean_original:.2f}s, std={lap_std_original:.2f}s")
        
        if val_mae_norm > obj.mae * 1.5:
            print(f"\n⚠️  WARNING: Validation MAE is {val_mae_norm/obj.mae:.1f}x higher than training MAE!")
            print("    Model may be overfitting. Consider:")
            print("    - Increasing regularization (reg_alpha, reg_lambda)")
            print("    - Reducing model complexity (max_depth, n_estimators)")
            print("    - Adding more training data from diverse races")
        elif val_mae_norm < obj.mae * 1.1:
            print("\n✓ Model generalizes well to validation data")
        
        print("="*60 + "\n")
    else:
        print("\n⚠️  No base pace laps found in validation data after filtering")
    
    # Label validation data
    print("\nLabeling validation data...")
    obj1 = BasePaceModel(df1)
    labeled_df = label_modes_with_confidence(obj1.df, obj.model, obj1.X, confidence_threshold=1.5)
    
    
    # Write to Unity Catalog Silver layer
    print("\nWriting labeled validation data to Unity Catalog...")
    labeled_df_output = labeled_df.drop(columns=['predicted_base_pace', 'delta'])
    spark_df = spark.createDataFrame(labeled_df_output)
    spark_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.f1_racing_laptime_pred.silver_validating")
    print("✓ Labeled validation data saved to workspace.f1_racing_laptime_pred.silver_validating")
    
    # Now read full training data for labeling (only needed at the end)
    print("\nReading full training data for labeling...")
    df = spark.table("workspace.f1_racing_laptime_pred.raw_training").toPandas()
    
    print("Labeling training data...")
    obj_train = BasePaceModel(df)
    labeled_train_df = label_modes_with_confidence(obj_train.df, obj.model, obj_train.X, confidence_threshold=1.5)
    
    print("\nWriting labeled training data to Unity Catalog...")
    labeled_train_df_output = labeled_train_df.drop(columns=['predicted_base_pace', 'delta'])
    spark_train_df = spark.createDataFrame(labeled_train_df_output)
    spark_train_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.f1_racing_laptime_pred.silver_training")
    print("✓ Labeled training data saved to workspace.f1_racing_laptime_pred.silver_training")
    
    print(f"\n{'='*60}")
    print("COMPLETE! Silver layer tables created successfully.")
    print(f"{'='*60}")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use the training data features from the model
print("=== CORRELATION ANALYSIS ===")
print(f"\nAnalyzing {len(obj.features)} features from trained model\n")

# Get feature matrix and target
X_corr = obj.X
y_corr = obj.y

# Compute correlation matrix for all features
corr_matrix = X_corr.corr()

# 1. Find highly correlated feature pairs (potential redundancy)
print("\n" + "="*60)
print("HIGHLY CORRELATED FEATURE PAIRS (|correlation| > 0.8)")
print("="*60)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    for feat1, feat2, corr_val in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True):
        print(f"{feat1:30s} <-> {feat2:30s} : {corr_val:6.3f}")
    print(f"\n⚠️  Found {len(high_corr_pairs)} highly correlated pairs")
    print("Consider removing one feature from each pair to reduce multicollinearity")
else:
    print("✓ No highly correlated feature pairs found (all |correlation| < 0.8)")

# 2. Correlation with target variable
print("\n" + "="*60)
print("TOP 15 FEATURES CORRELATED WITH LAP TIME")
print("="*60)

# Compute correlation with target
target_corr = pd.DataFrame({
    'feature': X_corr.columns,
    'correlation': [X_corr[col].corr(y_corr) for col in X_corr.columns]
}).sort_values('correlation', key=abs, ascending=False)

print(target_corr.head(15).to_string(index=False))

# 3. Compare with XGBoost feature importance
print("\n" + "="*60)
print("CORRELATION vs FEATURE IMPORTANCE (Top 10)")
print("="*60)

feature_importance = pd.DataFrame({
    'feature': obj.features,
    'importance': obj.model.feature_importances_
}).sort_values('importance', ascending=False).head(10)

comparison = feature_importance.merge(
    target_corr, on='feature', how='left'
)
print(comparison.to_string(index=False))

# 4. Visualization: Heatmap of top correlated features
print("\n" + "="*60)
print("CORRELATION HEATMAP (Top 20 features by importance)")
print("="*60)

top_20_features = obj.model.feature_importances_.argsort()[-20:][::-1]
top_20_names = [obj.features[i] for i in top_20_features]

# Create correlation matrix for top features
top_corr = X_corr[top_20_names].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(top_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap: Top 20 Features by Importance', fontsize=14, pad=20)
plt.tight_layout()
display(plt.gcf())
plt.close()

# 5. Recommendations
print("\n" + "="*60)
print("RECOMMENDATIONS")
print("="*60)

if high_corr_pairs:
    print("\n1. MULTICOLLINEARITY DETECTED:")
    print("   Consider removing these features (keep the one with higher importance):")
    for feat1, feat2, corr_val in high_corr_pairs[:5]:  # Show top 5
        imp1 = obj.model.feature_importances_[obj.features.index(feat1)]
        imp2 = obj.model.feature_importances_[obj.features.index(feat2)]
        keep = feat1 if imp1 > imp2 else feat2
        remove = feat2 if imp1 > imp2 else feat1
        print(f"   - Remove '{remove}' (keep '{keep}', importance: {max(imp1, imp2):.3f})")
else:
    print("\n1. ✓ No significant multicollinearity detected")

low_importance = feature_importance[feature_importance['importance'] < 0.01]
if len(low_importance) > 0:
    print(f"\n2. LOW IMPORTANCE FEATURES:")
    print(f"   {len(low_importance)} features have <1% importance")
    print(f"   Consider removing to simplify model")
else:
    print("\n2. ✓ All features contribute meaningfully (>1% importance)")

print(f"\n3. CURRENT MODEL:")
print(f"   - Total features: {len(obj.features)}")
print(f"   - Training MAE: {obj.mae:.3f} (normalized)")
print(f"   - After removing redundant features, retrain and check if validation MAE improves")